# 06 — Preparar SUN RGB-D + HomeObjects automaticamente

Execute este notebook no **Windows/Jupyter**, dentro da pasta `notebooks` do projeto.
Copie antes as pastas `src` e `notebooks` deste pacote para sua pasta do TCC.

O ajuste fino continuará a partir do **modelo antigo**, sem alterar as 12 classes
ou a política V2. A SUN reforça especialmente camas, cadeiras, mesas e sofás;
`cabinet` não será convertido em `wardrobe`. A profundidade não é utilizada
neste experimento RGB/2D. Ganho de precisão ainda precisa ser medido.

Esta etapa baixa anotações e imagens, valida as caixas nas dimensões reais,
preserva Home train/val e TEST e gera o pacote para o Colab. Não edite planilhas
nem preencha rótulos. O TEST oficial SUN permanece fora deste experimento.

Fonte: https://rgbd.cs.princeton.edu/


## 1. Dependências e caminhos

Não precisa de GPU aqui. O download oficial de imagens é grande, mesmo extraindo
apenas o RGB necessário. Reserve cerca de **15 GB livres** na unidade do projeto;
o notebook também verifica espaço antes de baixar e extrair.
Os caminhos abaixo usam sua pasta real e a base validada pelo notebook 02.


In [ ]:
%pip install -q scipy pandas pillow pyyaml matplotlib seaborn


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display

RAIZ_PROJETO = Path(r"C:\Users\diego\OneDrive\Documentos_One_Drive\TCC\tcc_visao_computacional")
DATASET_HOME = RAIZ_PROJETO / "data/processed/homeobjects_yolo_automatico/20261013_000954_320030"
MODELO_ANTIGO = RAIZ_PROJETO / "models/yolov8n_baseline/best.pt"
PASTA_RAW_SUN = RAIZ_PROJETO / "data/raw/sunrgbd"
SEMENTE = 42
PROPORCAO_SUN_VAL = 0.20
SHA_MODELO_ANTIGO = "4170653e0c0a604f85b6747e42b3fd2ff5ddb765ee4762915dfc9f3d33d322d9"

for p in [DATASET_HOME / "data.yaml", MODELO_ANTIGO,
          RAIZ_PROJETO / "src/preparacao_sunrgbd.py",
          RAIZ_PROJETO / "src/validacao_alertas.py"]:
    if not p.is_file():
        raise FileNotFoundError(f"Arquivo não encontrado: {p}. Confira a cópia do pacote e o caminho.")
sys.path.insert(0, str(RAIZ_PROJETO))
from src.auditoria_dados import _calcular_sha256
from src.preparacao_sunrgbd import (
    ALIASES, AMBIGUAS, obter_metadados, inventariar, candidatos_treino,
    obter_imagens, criar_base_mista, gerar_previa,
)

if _calcular_sha256(MODELO_ANTIGO) != SHA_MODELO_ANTIGO:
    raise RuntimeError("O best.pt difere do modelo antigo avaliado no 05d. Confira o arquivo; não substitua o hash para ocultar a diferença.")
print("Modelo antigo confirmado. Nenhum peso será modificado nesta etapa.")
print("HomeObjects de desenvolvimento:", DATASET_HOME)
print("SUN RGB-D raw:", PASTA_RAW_SUN)


## 2. Ler metadados e a divisão oficial

A associação usa o caminho completo da cena em `sequenceName/rgbpath`, incluindo
o nome original da imagem. Não depende da ordem de um pacote `img-000001.jpg`.
O `allsplit.mat` decide quem pertence ao treino oficial; não usamos um corte
numérico de 5.050 registros como substituto da divisão.


In [ ]:
registros, divisao = obter_metadados(PASTA_RAW_SUN)
inventario = inventariar(registros, divisao)
print("Registros anotados:", len(registros))
display(inventario.loc[inventario.classe_sun.isin(
    ["bed", "chair", "table", "sofa", "armoire", "cabinet", "plant", "picture", "window"]
)])


## 3. Compatibilidade entre classes

Mantemos os IDs do HomeObjects. `armoire → wardrobe` é uma equivalência de
guarda-roupa; `cabinet`, `dresser` e `bookshelf` permanecem outras categorias.
`plant`, `picture`, `painting`, `sofa_chair`, `desk` e `computer` são ambíguas
para esta primeira integração: suas imagens ficam fora do experimento para
evitar ensinar alvos sem rótulo como fundo. É um recorte conservador, não uma
afirmação de que essas categorias sejam inválidas em outros projetos.

As classes que não recebem exemplos SUN continuam sendo treinadas com HomeObjects.


In [ ]:
display(pd.DataFrame(sorted(ALIASES.items()), columns=["classe_sun_normalizada", "classe_projeto"]))
candidatos, exclusoes_iniciais = candidatos_treino(registros, divisao)
print("Candidatos do TREINO oficial:", len(candidatos))
print("Exclusões iniciais registradas:", len(exclusoes_iniciais))
print("Categorias ambíguas:", sorted(AMBIGUAS))


## 4. Baixar e extrair o RGB

O programa retoma downloads `.part` quando o servidor suporta isso. Se você já
extraiu o ZIP oficial, mantenha a estrutura `data/raw/sunrgbd/SUNRGBD/...`; as
imagens existentes são reutilizadas. Não coloque arquivos de um pacote de teste
renomeado nesta pasta. Profundidade e `fullres` não são extraídos.


In [ ]:
obter_imagens(PASTA_RAW_SUN, candidatos)
print("Download e extração RGB concluídos.")


## 5. Converter, agrupar, preservar TEST e exportar

Cada imagem é aberta para obter largura/altura reais. Caixas `[x,y,w,h]` de
origem MATLAB são convertidas para YOLO; bordas excedentes são recortadas
nas dimensões reais. Um alvo inválido exclui a imagem inteira e é registrado
no relatório. Não inventamos caixas, classes nem gabaritos.

Somente o treino oficial SUN é separado em aproximadamente 80% treino e 20%
validação, por grupos de captura quando identificáveis e hashes iguais.
Grupos SUN relacionados ao Home são excluídos. Home train/val mantém sua divisão;
as 403 imagens e anotações TEST mantêm seus caminhos e bytes. Esses hashes e
caminhos não recuperam toda semelhança visual nem todas as salas físicas.


In [ ]:
DATASET_MISTO, ZIP_COLAB, resumo = criar_base_mista(
    DATASET_HOME, PASTA_RAW_SUN, RAIZ_PROJETO, registros, divisao,
    proporcao_sun_val=PROPORCAO_SUN_VAL, semente=SEMENTE,
)
print(json.dumps(resumo, ensure_ascii=False, indent=2))
display(pd.DataFrame(resumo["imagens_por_origem"]))


## 6. Conferir a distribuição e os exemplos automáticos

As contagens abaixo refletem o que realmente entrou na base após as exclusões.
Os números exploratórios da POC que assumiam 640×480 não são reutilizados.
Esta visualização é uma conferência de conversão, não um preenchimento de planilha.


In [ ]:
RELATORIOS = Path(resumo["relatorios"])
distribuicao = pd.read_csv(RELATORIOS / "distribuicao_classes_mista.csv")
display(distribuicao.pivot_table(index=["id_classe", "classe"], columns=["split", "origem"], values="objetos", fill_value=0))
sun_final = pd.read_csv(RELATORIOS / "grupos_sun_e_splits.csv")
previas = gerar_previa(sun_final, RELATORIOS / "previas")
import matplotlib.pyplot as plt
from PIL import Image
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for ax in axes.flat:
    ax.axis("off")
for ax, p in zip(axes.flat, previas):
    with Image.open(p) as im:
        ax.imshow(im)
    ax.set_title(p.name)
plt.tight_layout()
plt.show()


## 7. Arquivos para o próximo passo

Disponibilizar o ZIP abaixo no Drive, na pasta `TCC_Visao_Computacional/datasets`.
Copie o modelo antigo para `TCC_Visao_Computacional/modelos/yolov8n_baseline/best.pt`.
Depois execute `07_ajuste_fino_sunrgbd_colab.ipynb` no Colab com GPU T4.

A revisão desta etapa utiliza `resumo_preparacao_sunrgbd.json`,
`distribuicao_classes_mista.csv`, `exclusoes_automaticas.csv` e o notebook rodado.
A análise é realizada a partir dos relatórios exportados.


In [ ]:
print("ZIP para o Colab:", ZIP_COLAB)
print("Resumo:", RELATORIOS / "resumo_preparacao_sunrgbd.json")
print("Distribuição:", RELATORIOS / "distribuicao_classes_mista.csv")
print("Exclusões:", RELATORIOS / "exclusoes_automaticas.csv")
print("Modelo antigo para copiar ao Drive:", MODELO_ANTIGO)
